# Задание 2: Сканирование хромосомы человека

Используем `Bio.motifs` для поиска сайтов связывания на chr1.

In [14]:
from Bio import motifs, SeqIO
from Bio.Seq import Seq

## 1. Создаём мотив через Bio.motifs

In [15]:
sites = [
    "GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA",
    "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC",
    "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
    "CAGGTTGGC"
]

instances = [Seq(s) for s in sites]
m = motifs.create(instances)

# Задаём фоновые частоты генома человека
m.background = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}

print(f"Консенсус: {m.consensus}")
print(f"Длина мотива: {len(m)}")
print("\nPSSM (PWM):")
print(m.pssm)

Консенсус: CAGGTCAGC
Длина мотива: 9

PSSM (PWM):
        0      1      2      3      4      5      6      7      8
A:  -1.56   1.44  -1.56   -inf   -inf  -0.56   1.25  -0.56  -1.56
C:   1.55  -0.04  -1.04   -inf   -inf   1.55   -inf   -inf   1.96
G:  -1.04   -inf   1.96   2.29   -inf   -inf   0.55   1.96   -inf
T:  -0.56   -inf   -inf   -inf   1.76  -0.56   -inf   -inf  -1.56



## 2–3. берём первый миллион нуклеотидов

In [16]:

CHR1_PATH = "GCA_029856635.1_ASM2985663v1_genomic.fna"

record = next(SeqIO.parse(CHR1_PATH, "fasta"))
seq = record.seq[:1_000_000]

print(f"Длина фрагмента: {len(seq)} нт")

Длина фрагмента: 37163 нт


## 4–5. Поиск на прямой цепи, скор > 5.0

In [17]:
THRESHOLD = 5.0

hits_forward = []
for pos, score in m.pssm.search(seq, threshold=THRESHOLD):
    hits_forward.append((pos, '+', round(score, 4)))

print(f"Хиты на прямой цепи (скор > {THRESHOLD}): {len(hits_forward)}")
for pos, strand, score in hits_forward[:20]:
    print(f"  pos={pos}, strand={strand}, score={score}")

Хиты на прямой цепи (скор > 5.0): 44
  pos=-35918, strand=+, score=5.867700099945068
  pos=1729, strand=+, score=7.505099773406982
  pos=2887, strand=+, score=5.867700099945068
  pos=3295, strand=+, score=6.3927998542785645
  pos=4694, strand=+, score=6.980000019073486
  pos=-29298, strand=+, score=6.090099811553955
  pos=-29206, strand=+, score=6.807799816131592
  pos=10227, strand=+, score=5.454999923706055
  pos=12838, strand=+, score=6.980000019073486
  pos=13131, strand=+, score=6.980000019073486
  pos=13207, strand=+, score=6.150000095367432
  pos=-23299, strand=+, score=6.090099811553955
  pos=-23251, strand=+, score=7.565000057220459
  pos=14218, strand=+, score=7.090099811553955
  pos=14599, strand=+, score=6.090099811553955
  pos=16921, strand=+, score=9.030200004577637
  pos=17073, strand=+, score=5.090099811553955
  pos=19037, strand=+, score=5.867700099945068
  pos=-15744, strand=+, score=5.867700099945068
  pos=22024, strand=+, score=7.565000057220459


## 6. Поиск на обратной комплементарной цепи

Транскрипционные факторы могут связываться на обеих цепях ДНК.

In [18]:
seq_rc = seq.reverse_complement()

hits_reverse = []
for pos, score in m.pssm.search(seq_rc, threshold=THRESHOLD):
    orig_pos = len(seq) - pos - len(m)
    hits_reverse.append((orig_pos, '-', round(score, 4)))

print(f"Хиты на обратной цепи (скор > {THRESHOLD}): {len(hits_reverse)}")
for pos, strand, score in hits_reverse[:20]:
    print(f"  pos={pos}, strand={strand}, score={score}")

Хиты на обратной цепи (скор > 5.0): 44
  pos=36866, strand=-, score=6.282700061798096
  pos=35122, strand=-, score=6.090099811553955
  pos=34389, strand=-, score=9.61520004272461
  pos=33535, strand=-, score=7.565000057220459
  pos=32192, strand=-, score=10.090100288391113
  pos=32082, strand=-, score=6.867700099945068
  pos=31491, strand=-, score=6.282700061798096
  pos=68397, strand=-, score=7.090099811553955
  pos=30383, strand=-, score=9.392800331115723
  pos=67431, strand=-, score=7.505099773406982
  pos=67148, strand=-, score=7.980000019073486
  pos=29849, strand=-, score=6.565000057220459
  pos=28864, strand=-, score=9.090100288391113
  pos=28811, strand=-, score=7.505099773406982
  pos=65685, strand=-, score=7.505099773406982
  pos=65365, strand=-, score=6.565000057220459
  pos=64614, strand=-, score=5.454999923706055
  pos=64507, strand=-, score=7.917900085449219
  pos=62597, strand=-, score=8.090100288391113
  pos=25176, strand=-, score=5.867700099945068


In [19]:
all_hits = hits_forward + hits_reverse
all_hits.sort(key=lambda x: x[0])

print(f"Всего хитов: {len(all_hits)}")

with open("hits_chr1.txt", "w") as f:
    f.write("position\tstrand\tscore\n")
    for pos, strand, score in all_hits:
        f.write(f"{pos}\t{strand}\t{score}\n")


Всего хитов: 88
